In [ ]:
import pandas as pd

df = pd.read_parquet("data/train-00000-of-00001.parquet")
df.shape # (rows, columns)
df.head() # the first 5 films
df.info() # column names, types, missing values

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
df = df[(df["positive"] + df["negative"]) >= 10]
df["year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year
df["score"] = df["positive"] / (df["positive"] + df["negative"]) * 100
threshold = df.groupby("year")["score"].transform("quantile", 0.75)
df["hit"] = df["score"] >= threshold
df["language_count"] = df["supported_languages"].str.len().fillna(0).astype(int)

In [33]:
from sklearn.model_selection import train_test_split
df_tmp, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_tmp, test_size=0.25, random_state=42)
len(df_train), len(df_val), len(df_test)

(33993, 11332, 11332)

In [34]:
features = [
    "price", "required_age","language_count"
]

X_train, y_train = df_train[features], df_train["hit"]
X_val,   y_val   = df_val[features],   df_val["hit"]
X_test,  y_test  = df_test[features],  df_test["hit"]

In [35]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
model = LogisticRegression(max_iter=1000,class_weight="balanced")
model.fit(X_train, y_train)
pred_baseline = baseline.predict(X_val)
pred_model = model.predict(X_val)
print("Baseline:", accuracy_score(y_val, pred_baseline), recall_score(y_val, pred_baseline))
print("Model: ", accuracy_score(y_val, pred_model), recall_score(y_val, pred_model))

Baseline: 0.7534415813625133 0.0
Model:  0.42507942110836566 0.67680744452398
